In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta

year = "2024"

In [2]:
standings_url =  "https://fbref.com/en/comps/31/Liga-MX-Stats"

In [3]:
#standings_url =  'https://fbref.com/en/comps/31/schedule/Liga-MX-Scores-and-Fixtures'

In [4]:
years = list(range(2024, 2022, -1))
all_matches = []

In [5]:
data = requests.get(standings_url)
soup = BeautifulSoup(data.text)
standings_table = soup.select('table.stats_table')[0]

In [ ]:

for year in years:
    data = requests.get(standings_url)
    soup = BeautifulSoup(data.text)
    standings_table = soup.select('table.stats_table')[0]

    links = [l.get("href") for l in standings_table.find_all('a')]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url = f"https://fbref.com{previous_season}"
    
    time.sleep(1)
    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        data = requests.get(team_url)
        matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
        soup = BeautifulSoup(data.text)
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        data = requests.get(f"https://fbref.com{links[0]}")
        shooting = pd.read_html(data.text, match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        team_data = team_data[team_data["Comp"] == "Liga MX"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)
        time.sleep(12)#tiene este tiempo para cumplir con las politicas de la pagina web

C:\Users\edgar\AppData\Local\Temp\ipykernel_27136\2676438321.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_27136\2676438321.py:22: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  shooting = pd.read_html(data.text, match="Shooting")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_27136\2676438321.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_27136\2676438321.py:22: FutureWarning: Passing literal 

In [ ]:
len(all_matches)

In [ ]:
match_df = pd.concat(all_matches)

In [ ]:
match_df.columns = [c.lower() for c in match_df.columns]

In [ ]:
match_df

Extraer partidos de la semana

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta

# Obtener tabla
standings_url =  'https://fbref.com/en/comps/31/schedule/Liga-MX-Scores-and-Fixtures'
data = requests.get(standings_url)
soup = BeautifulSoup(data.text)
standings_table = soup.select('table.stats_table')[0]

# Extraer los datos
matches = []
rows = soup.find_all("tr")  # Encontrar todas las filas de la tabla

for row in rows:
    home_team = row.find("td", {"data-stat": "home_team"})
    away_team = row.find("td", {"data-stat": "away_team"})
    date = row.find("td", {"data-stat": "date"})
    time = row.find("td", {"data-stat": "start_time"})

    # Si la hora no está disponible, asignamos un NA
    if time is None or time.text.strip() == "NA":
        match_time = pd.NaT
    else:
        # Solo intentamos acceder al texto si time no es None
        match_time = time.find("span").text.strip() if time.find("span") else pd.NaT
        # Truncar la fecha a solo día, mes y año (sin hora)
        try:
            match_date = datetime.strptime(date_text.split()[0], "%Y-%m-%d")
        except ValueError:
            match_date = pd.NaT  # Asignamos NA si la fecha no es válida
        

    # Verificar si los datos de los equipos y la fecha están disponibles
    if home_team and away_team and date:
        matches.append({
            "team": home_team.text.strip(),
            "opponent": away_team.text.strip(),
            "date": date.text.strip(),
            "venue": "Home" if home_team else "Away",  # Determinamos si es local o visitante
            "time": match_time
        })
# Convertir los datos a un DataFrame de pandas para generar la tabla
df = pd.DataFrame(matches)

# Convertir las fechas de string a formato datetime
df['date'] = pd.to_datetime(df['date'])

# Calcular el rango de fechas de la próxima semana
hoy = datetime.now()
una_semana_despues = hoy + timedelta(days=7)

# Filtrar los partidos de la próxima semana
df_proxima_semana = df[(df['date'] > hoy) & (df['date'] <= una_semana_despues)]
df_proxima_semana = df_proxima_semana.drop_duplicates()


# Convertir los datos a un DataFrame de pandas para generar la tabla
df = pd.DataFrame(matches)

# Convertir las fechas de string a formato datetime
df['date'] = pd.to_datetime(df['date'])

# Calcular el rango de fechas de la próxima semana
hoy = datetime.now()
una_semana_despues = hoy + timedelta(days=7)

# Filtrar los partidos de la próxima semana
df_proxima_semana = df[(df['date'] > hoy) & (df['date'] <= una_semana_despues)]
df_proxima_semana = df_proxima_semana.drop_duplicates()

# Mostrar la tabla filtrada
print(df_proxima_semana)

Agregarlos la talba de estadisiticas

In [ ]:
df_ready_for_predictions = pd.concat([df_proxima_semana, match_df], ignore_index=True, sort=False)

In [ ]:
df_ready_for_predictions

In [ ]:
df_ready_for_predictions.to_csv("matchesSemiFinals.csv")

In [ ]:
standings_url =  'https://fbref.com/en/comps/31/schedule/Liga-MX-Scores-and-Fixtures'
data = requests.get(standings_url)
soup = BeautifulSoup(data.text)

# Encontrar todos los elementos de los partidos
matchup = soup.find_all(class_="matchup")

for match in matchup:
    # Extraer los equipos locales y visitantes
    local_teams = match.find_all(class_="matchup-team team1")
    visit_teams = match.find_all(class_="matchup-team team2")
    raw_dates = match.find_all(class_="match-date")

local_teams_names =[]
visit_teams_names =[]
match_dates = []
for team in local_teams:
    name = team.find('a').text.strip()  # Buscar el texto dentro de la etiqueta <a>
    local_teams_names.append(name)

for team in visit_teams:
    name = team.find('a').text.strip()  
    visit_teams_names.append(name)
    
for match in raw_dates:
    date = match.text.strip()
    match_dates.append(date)
match_dates =  match_dates+ match_dates # para que coincida tamaño y orden con las otra listas extendidas
    # Cambiar a formato compatible para pandas
match_dates = [date.replace('\xa0', ' ') for date in match_dates]
match_dates = [ date+" 2024"  for date in match_dates]

match_dates = pd.to_datetime(match_dates).date


for team in visit_teams_names:
    if team not in local_teams_names:
        local_teams_names.append(team)
for team in local_teams_names:
    if team not in visit_teams_names:
        visit_teams_names.append(team)


future_matches_df = pd.DataFrame({
    'team': local_teams_names,
    'opponent': visit_teams_names,
    'date': match_dates,
    'venue':["Home","Home","Away","Away"], # falta automatizar esto
    'time': ["17:05","19:00","17:05","19:00"] # falta automatizar esto
})
